# TinyLlama Alpaca LoRA Fine-Tuning — Clean Google Colab Notebook

This notebook is a cleaned version of the uploaded TinyLlama + Alpaca + LoRA workflow.

### Main fixes
- Avoids the CPU/CUDA input mismatch during generation.
- Uses `max_new_tokens` without conflicting `max_length` generation settings.
- Uses `warmup_steps` instead of deprecated `warmup_ratio`.
- Does not install unnecessary `torchao`.
- Keeps LoRA adapter saving and optional merged-model saving.
- Includes baseline, training, qualitative, and held-out evaluation.

In [2]:
# Cell 1 — Install dependencies

!pip install -q -U transformers datasets accelerate peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 22.1 MB/s eta 0:00:00


In [3]:
print('Upgrading torchao to a compatible version...')
!pip install -U torchao
print('torchao upgraded.')

Upgrading torchao to a compatible version...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 36.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
torchao upgraded.


In [4]:
# Cell 2 — Check Colab environment

import torch
import transformers

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Enable a GPU runtime in Colab for practical fine-tuning.")

Transformers: 5.18.0
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [5]:
# Cell 3 — Configuration

MODEL_ID = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
DATASET_NAME = "tatsu-lab/alpaca"

SEED = 42
EVAL_SPLIT_SIZE = 0.02

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]

# Training
OUTPUT_DIR = "/content/tinyllama-alpaca-lora"
ADAPTER_DIR = "/content/tinyllama-alpaca-lora-adapter"
MERGED_DIR = "/content/tinyllama-alpaca-merged"

NUM_TRAIN_EPOCHS = 1
PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03

MAX_SEQ_LENGTH = 512

LOGGING_STEPS = 50
EVAL_STEPS = 500
SAVE_STEPS = 500
SAVE_TOTAL_LIMIT = 2

# Generation
MAX_NEW_TOKENS = 150
TEMPERATURE = 0.7
TOP_P = 0.9

# Run controls
RUN_BASELINE_EVAL = True
DO_TRAIN = True
RUN_FINETUNED_QUALITATIVE_EVAL = True
RUN_FINETUNED_HELDOUT_EVAL = True

NUM_HELDOUT_EVAL_SAMPLES = 10

SAVE_ADAPTER = True
SAVE_MERGED_MODEL = True

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Configuration loaded.")

Configuration loaded.


In [6]:
# Cell 4 — Test prompts

TEST_PROMPTS = [
    "Hello! What can you help me with?",
    "Explain what artificial intelligence is in simple words.",
    "Write a short Python function that adds two numbers.",
    "What is the capital of India?",
    "Give me three tips for learning programming.",
    "Explain the difference between RAM and storage.",
    "Write a short friendly message saying good morning.",
    "Why is the sky blue?",
]

print("Number of test prompts:", len(TEST_PROMPTS))

Number of test prompts: 8


In [7]:
# Cell 5 — Load tokenizer

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

EOS token: </s>
PAD token: </s>


In [8]:
# Cell 6 — Load base model for baseline evaluation

from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.pad_token_id = tokenizer.pad_token_id

# Prevent an old/default generation max_length from conflicting
# with max_new_tokens during generation.
if hasattr(model, "generation_config"):
    model.generation_config.max_length = None

model.eval()

print("Model loaded.")
print("Parameters:", model.num_parameters())
print("Device:", model.device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 4.40GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Model loaded.
Parameters: 1100048384
Device: cuda:0


In [9]:
# Cell 7 — Correct baseline generation function

def generate_response(
    prompt,
    tokenizer,
    model,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

    # IMPORTANT: move inputs to the same device as the model
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

In [10]:
# Cell 8 — Baseline evaluation

if RUN_BASELINE_EVAL:
    for i, prompt in enumerate(TEST_PROMPTS, start=1):
        print("\n" + "=" * 80)
        print(f"TEST {i}")
        print("\nUSER:")
        print(prompt)

        response = generate_response(prompt, tokenizer, model)

        print("\nBASE MODEL:")
        print(response)


TEST 1

USER:
Hello! What can you help me with?

BASE MODEL:
I have a `s3://mybucket/folder/file.json` and I would like to download it to a local file `file.json`

I tried:
```
s3cmd get s3://mybucket/folder/file.json file.json
```

but it just downloads the file, not the contents. It seems like it's just downloading the file, without the contents.

I tried using `s3cmd put`, but I just get the error: `s3cmd: invalid argument`

Is there a way to do what I want?
<issue_comment>username_1

TEST 2

USER:
Explain what artificial intelligence is in simple words.

BASE MODEL:
I am a mechanical engineer and want to work on AI.
Is AI a good career?
What is the difference between AI and deep learning?
What is the difference between AI and machine learning?
What is the difference between AI and machine intelligence?
What is the difference between AI and machine learning?
What is the difference between AI and computer vision?
What is the difference between AI and machine learning?
What is the di

In [11]:
# Cell 9 — Load Alpaca dataset

from datasets import load_dataset

dataset = load_dataset(DATASET_NAME)

print(dataset)
print("Columns:", dataset["train"].column_names)
print("Training examples:", len(dataset["train"]))
print("\nFirst example:")
print(dataset["train"][0])

README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…): reconstructing file:   0%|          |  0.00B / 24.2MB            

data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 52002
    })
})
Columns: ['instruction', 'input', 'output', 'text']
Training examples: 52002

First example:
{'instruction': 'Give three tips for staying healthy.', 'input': '', 'output': '1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.', 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'}


In [12]:
# Cell 10 — Train/evaluation split

split_dataset = dataset["train"].train_test_split(
    test_size=EVAL_SPLIT_SIZE,
    seed=SEED,
)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("Train examples:", len(train_dataset))
print("Eval examples:", len(eval_dataset))

Train examples: 50961
Eval examples: 1041


In [13]:
# Cell 11 — Free baseline model before training

del model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Baseline model removed from memory.")

Baseline model removed from memory.


In [14]:
# Cell 12 — LoRA configuration

from peft import LoraConfig

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

print(lora_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.2', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'q_proj', 'o_proj', 'up_proj', 'gate_proj', 'v_proj', 'k_proj', 'down_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_ty

In [15]:
# Cell 13 — Reload model for LoRA training

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

print("Training model loaded.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Training model loaded.


In [16]:
# Cell 14 — Define chat template

tokenizer.chat_template = '''{% for message in messages %}
{% if message["role"] == "user" %}
### User:
{{ message["content"] }}
{% elif message["role"] == "assistant" %}
### Assistant:
{{ message["content"] }}{{ eos_token }}
{% endif %}
{% endfor %}
{% if add_generation_prompt %}
### Assistant:
{% endif %}'''

print(tokenizer.chat_template)

{% for message in messages %}
{% if message["role"] == "user" %}
### User:
{{ message["content"] }}
{% elif message["role"] == "assistant" %}
### Assistant:
{{ message["content"] }}{{ eos_token }}
{% endif %}
{% endfor %}
{% if add_generation_prompt %}
### Assistant:
{% endif %}


In [17]:
# Cell 15 — Convert Alpaca examples to messages

def alpaca_to_messages(example):
    instruction = example["instruction"].strip()
    user_input = example["input"].strip()
    output = example["output"].strip()

    user_content = instruction
    if user_input:
        user_content += f"\n\n{user_input}"

    return {
        "messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output},
        ]
    }

In [18]:
# Cell 16 — Format datasets

train_dataset = train_dataset.map(
    alpaca_to_messages,
    remove_columns=train_dataset.column_names,
)

eval_dataset = eval_dataset.map(
    alpaca_to_messages,
    remove_columns=eval_dataset.column_names,
)

print("Train example:")
print(train_dataset[0])

print("\nEval example:")
print(eval_dataset[0])

Map:   0%|          | 0/50961 [00:00<?, ? examples/s]

Map:   0%|          | 0/1041 [00:00<?, ? examples/s]

Train example:
{'messages': [{'content': 'Given two different texts, compare them.\n\nText A: The cost of living has increased significantly over the past decade.\nText B: The cost of living has decreased significantly in the past decade.', 'role': 'user'}, {'content': 'Text A and Text B contrast in their opinion of the cost of living in the past decade. Text A states that the cost of living has increased significantly while Text B claims that the cost of living has decreased significantly.', 'role': 'assistant'}]}

Eval example:
{'messages': [{'content': 'What would be the best type of exercise for a person who has arthritis?', 'role': 'user'}, {'content': 'For someone with arthritis, the best type of exercise would be low-impact activities like yoga, swimming, or walking. These exercises provide the benefits of exercise without exacerbating the symptoms of arthritis.', 'role': 'assistant'}]}


In [19]:
# Cell 17 — Check chat template

formatted_example = tokenizer.apply_chat_template(
    train_dataset[0]["messages"],
    tokenize=False,
)

print(formatted_example)

### User:
Given two different texts, compare them.

Text A: The cost of living has increased significantly over the past decade.
Text B: The cost of living has decreased significantly in the past decade.
### Assistant:
Text A and Text B contrast in their opinion of the cost of living in the past decade. Text A states that the cost of living has increased significantly while Text B claims that the cost of living has decreased significantly.</s>



In [20]:
# Cell 18 — Calculate warmup steps

import math

steps_per_epoch = math.ceil(
    len(train_dataset)
    / PER_DEVICE_TRAIN_BATCH_SIZE
    / GRADIENT_ACCUMULATION_STEPS
)

total_training_steps = steps_per_epoch * NUM_TRAIN_EPOCHS

warmup_steps = math.ceil(
    total_training_steps * WARMUP_RATIO
)

print("Steps per epoch:", steps_per_epoch)
print("Total training steps:", total_training_steps)
print("Warmup steps:", warmup_steps)

Steps per epoch: 3186
Total training steps: 3186
Warmup steps: 96


In [21]:
# Cell 19 — SFT configuration

from trl import SFTConfig

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_TRAIN_EPOCHS,

    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,

    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,

    warmup_steps=warmup_steps,

    max_length=MAX_SEQ_LENGTH,

    logging_steps=LOGGING_STEPS,

    eval_strategy="steps",
    eval_steps=EVAL_STEPS,

    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,

    fp16=torch.cuda.is_available(),

    seed=SEED,
    report_to="none",
)

print("SFT configuration created.")

SFT configuration created.


In [22]:
# Cell 20 — Create SFTTrainer

from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

print("SFTTrainer created successfully.")

Tokenizing train dataset:   0%|          | 0/50961 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/50961 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/50961 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/50961 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1041 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/1041 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1041 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/1041 [00:00<?, ? examples/s]

SFTTrainer created successfully.


In [23]:
# Cell 21 — Check trainable parameters

trainable_params = 0
all_params = 0

for _, param in trainer.model.named_parameters():
    all_params += param.numel()

    if param.requires_grad:
        trainable_params += param.numel()

trainable_percent = 100 * trainable_params / all_params

print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters: {all_params:,}")
print(f"Trainable percentage: {trainable_percent:.4f}%")

Trainable parameters: 12,615,680
Total parameters: 1,112,664,064
Trainable percentage: 1.1338%


In [24]:
# Cell 22 — Train

if DO_TRAIN:
    print("Starting training...")
    train_result = trainer.train()

    print("\nTraining completed.")
    print(train_result)
else:
    print("DO_TRAIN=False — training skipped.")

Starting training...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
500,1.302536,1.267076,1.280713,792892.000000,0.682847
1000,1.273129,1.247013,1.318102,1573583.000000,0.685912
1500,1.253388,1.236603,1.259951,2344069.000000,0.687286


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
500,1.302536,1.267076,1.280713,792892.000000,0.682847
1000,1.273129,1.247013,1.318102,1573583.000000,0.685912
1500,1.253388,1.236603,1.259951,2344069.000000,0.687286
2000,1.259794,1.230085,1.279077,3136918.000000,0.688299
2500,1.267651,1.224261,1.264506,3924248.000000,0.688940
3000,1.224527,1.221439,1.254713,4704509.000000,0.689566
3186,1.250965,1.221098,1.257174,4997160.000000,0.690016



Training completed.
TrainOutput(global_step=3186, training_loss=1.2702917061881756, metrics={'train_runtime': 10085.1526, 'train_samples_per_second': 5.053, 'train_steps_per_second': 0.316, 'total_flos': 4.167765507667968e+16, 'train_loss': 1.2702917061881756, 'epoch': 1.0})


In [25]:
# Cell 23 — Prepare fine-tuned model

finetuned_model = trainer.model
device = trainer.accelerator.device

finetuned_model.eval()

print("Fine-tuned model ready.")
print("Device:", device)

Fine-tuned model ready.
Device: cuda


In [26]:
# Cell 24 — Correct fine-tuned generation function

def generate_finetuned_response(
    messages,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = finetuned_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

In [27]:
# Cell 25 — Single fine-tuned prompt test

messages = [
    {
        "role": "user",
        "content": "Explain artificial intelligence in simple words."
    }
]

response = generate_finetuned_response(messages)

print("USER:")
print(messages[0]["content"])

print("\nFINE-TUNED MODEL:")
print(response)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER:
Explain artificial intelligence in simple words.

FINE-TUNED MODEL:
Artificial intelligence (AI) is the science and engineering of making computers and machines behave like humans in their decision-making, problem-solving and learning. It uses algorithms, software and data to simulate human behavior and reasoning. AI can be used to make decisions, solve problems, and generate creative ideas.


In [28]:
# Cell 26 — Qualitative evaluation

if RUN_FINETUNED_QUALITATIVE_EVAL:
    finetuned_results = []

    for i, prompt in enumerate(TEST_PROMPTS, start=1):
        messages = [
            {"role": "user", "content": prompt}
        ]

        response = generate_finetuned_response(messages)

        finetuned_results.append({
            "id": i,
            "prompt": prompt,
            "response": response,
        })

        print("\n" + "=" * 80)
        print(f"TEST {i}")
        print("\nUSER:")
        print(prompt)
        print("\nFINE-TUNED MODEL:")
        print(response)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TEST 1

USER:
Hello! What can you help me with?

FINE-TUNED MODEL:
Hello! I can help you with a variety of tasks, such as setting up a new computer, creating a new document, or troubleshooting a problem.


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TEST 2

USER:
Explain what artificial intelligence is in simple words.

FINE-TUNED MODEL:
Artificial Intelligence (AI) is the ability of machines to perform tasks that require human intelligence. It involves the development of software systems and algorithms that can be trained to perform a wide range of tasks such as speech recognition, image recognition, natural language processing, and more. AI can also be used for autonomous decision-making, such as recommending products, predicting customer behavior, and more.


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


TEST 5

USER:
Give me three tips for learning programming.

FINE-TUNED MODEL:
1. Set realistic goals for yourself and make a plan for achieving them.
2. Learn to think critically and make decisions based on data and evidence.
3. Take regular breaks and have a break from programming to refresh your mind and recharge.


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TEST 6

USER:
Explain the difference between RAM and storage.

FINE-TUNED MODEL:
RAM (Random Access Memory) is a type of memory that allows the computer to access information quickly and efficiently. It is used to store temporary data that is needed by the computer. Storage is a type of memory that stores permanent data, such as files or programs. It is used to store data that is not needed by the computer. RAM is faster than storage, as it is used to access data quickly, while storage is used to store data for later access.


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TEST 7

USER:
Write a short friendly message saying good morning.

FINE-TUNED MODEL:
Good morning! It's great to have you here.

TEST 8

USER:
Why is the sky blue?

FINE-TUNED MODEL:
The sky is blue because of the colors of the light, which reflect off of the particles in the atmosphere. The blue color is due to the scattering of light by the atoms and molecules in the atmosphere. The particles in the atmosphere absorb certain wavelengths of light, which are then reflected back to the observer, resulting in the blue color.


In [ ]:
# Cell 27 — Held-out Alpaca evaluation

if RUN_FINETUNED_HELDOUT_EVAL:
    import random

    random.seed(SEED)

    num_samples = min(
        NUM_HELDOUT_EVAL_SAMPLES,
        len(eval_dataset)
    )

    test_indices = random.sample(
        range(len(eval_dataset)),
        num_samples
    )

    eval_results = []

    for i, idx in enumerate(test_indices, start=1):
        example = eval_dataset[idx]

        user_message = next(
            m["content"]
            for m in example["messages"]
            if m["role"] == "user"
        )

        expected_answer = next(
            m["content"]
            for m in example["messages"]
            if m["role"] == "assistant"
        )

        generated_answer = generate_finetuned_response(
            [{"role": "user", "content": user_message}]
        )

        eval_results.append({
            "test_id": i,
            "dataset_index": idx,
            "input": user_message,
            "expected": expected_answer,
            "generated": generated_answer,
        })

        print("\n" + "=" * 80)
        print(f"TEST {i}")
        print("\nUSER:")
        print(user_message)
        print("\nEXPECTED:")
        print(expected_answer)
        print("\nMODEL:")
        print(generated_answer)

In [ ]:
# Cell 28 — Save LoRA adapter

import os

if SAVE_ADAPTER:
    os.makedirs(ADAPTER_DIR, exist_ok=True)

    trainer.model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)

    print("LoRA adapter saved successfully.")
    print("Location:", ADAPTER_DIR)

    print("\nSaved files:")
    for file in sorted(os.listdir(ADAPTER_DIR)):
        print(" -", file)
else:
    print("SAVE_ADAPTER=False — adapter saving skipped.")

In [ ]:
# Cell 29 — Merge and save standalone model

if SAVE_MERGED_MODEL:
    os.makedirs(MERGED_DIR, exist_ok=True)

    print("Merging LoRA weights...")

    merged_model = trainer.model.merge_and_unload()

    merged_model.save_pretrained(
        MERGED_DIR,
        safe_serialization=True
    )

    tokenizer.save_pretrained(MERGED_DIR)

    print("\nMerged model saved successfully.")
    print("Location:", MERGED_DIR)

    print("\nSaved files:")
    for file in sorted(os.listdir(MERGED_DIR)):
        print(" -", file)
else:
    print("SAVE_MERGED_MODEL=False — merged model saving skipped.")

In [32]:
# Cell 30 — Final verification

print("=" * 80)
print("FINAL VERIFICATION")
print("=" * 80)

print("\nModel:")
print(MODEL_ID)

print("\nDataset:")
print(DATASET_NAME)

print("\nTrain examples:")
print(len(train_dataset))

print("\nEval examples:")
print(len(eval_dataset))

print("\nLoRA adapter:")
print(ADAPTER_DIR)

print("\nMerged model:")
print(MERGED_DIR)

if os.path.exists(ADAPTER_DIR):
    print("\n✓ LoRA adapter directory exists")

if os.path.exists(MERGED_DIR):
    print("✓ Merged model directory exists")

print("\n✓ Notebook workflow completed.")

FINAL VERIFICATION

Model:
TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T

Dataset:
tatsu-lab/alpaca

Train examples:
50961

Eval examples:
1041

LoRA adapter:
/content/tinyllama-alpaca-lora-adapter

Merged model:
/content/tinyllama-alpaca-merged

✓ LoRA adapter directory exists
✓ Merged model directory exists

✓ Notebook workflow completed.
